# Day07 — финализация classification mini-product

## D7-01 — аудит handoff и read-only checkpoint

**Цель:** проверить готовность Day07 по SPEC §12. Входы — существующие Day02–Day05 references, Day06 bundle/manifest, consistency report и demo. Выход — inventory с SHA-256, checked metadata и явными missing paths. Данные настоящие, локальные; подготовки или восстановления нет.

**Краткая теория.** Config описывает алгоритм; fitted Pipeline хранит vocabulary/IDF и веса; cached predictions содержат ответы только на прежние строки. `load_inference_reference` проверяет provenance, схемы, IDs/order/labels, splits, CV choice и final cache. `load_bundle` дополнительно проверяет runtime, preprocessing implementation, metadata и fitted state. Равенство contracts связывает две проверки.

Путь проекта: **data → preprocessing → train/group-aware CV → frozen evaluation → inference**.
Путь новых текстов (только схема, здесь не запускается): raw `(N,)` → `minimal-v1` → fitted TF-IDF sparse `(N, V)` → classifier → predictions `(N,)`. Fit vocabulary/IDF разрешён только на train.

В этом пункте нет fit, transform/predict/score, CV, download, serialization или final evaluation. Чтение cached test outputs не создаёт новых test predictions. Загружается только доверенный локальный joblib: checksum проверяет целостность, но не автора.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src/finnews_sentiment").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/finnews_sentiment").is_dir():
    raise RuntimeError("Run from the project root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.models.inference_artifact import (
    InferenceBundle, InferenceReference, load_bundle, load_inference_reference,
)
print("Checkpoint: D7-01; local artifacts only; no preparation/model calls")

Checkpoint: D7-01; local artifacts only; no preparation/model calls


### Inventory: обязательные и опциональные входы

`required` означает вход принятого handoff для дальнейших задач Day07. Raw archive/canonical CSV и Day01 EDA outputs здесь optional: read-only loaders используют processed data и проверенную provenance, не запускают подготовку raw. Эти файлы понадобятся для отдельных сценариев воспроизведения. Missing required path останавливает checkpoint без автоматического download/rebuild.

Tracked/ignored ниже зафиксированы по Git inventory при создании checkpoint; существующая policy `.gitignore` не меняется. SHA-256 таблица показывает фактические bytes. Наличие файла и его новый hash сами по себе не доказывают корректность: semantic validators вызываются следующей ячейкой.


In [2]:
REQUIRED_GROUPS: dict[str, list[str]] = {'contracts/runtime': ['configs/dataset.json',
                       'configs/train_baseline.json',
                       'configs/experiments_day04.json',
                       'requirements-day01.txt',
                       'requirements-day03.txt',
                       'requirements-dev.txt'],
 'Day02 processed/provenance': ['data/processed/financial_phrasebank_75agree.csv',
                                'reports/day02/preprocessing.json',
                                'reports/day02/duplicates.json',
                                'reports/day02/provenance.json',
                                'reports/day02/observations.md'],
 'Day03 split/baseline': ['data/processed/day03/splits.csv',
                          'reports/day03/config.json',
                          'reports/day03/metrics.json',
                          'reports/day03/provenance.json',
                          'reports/day03/split_summary.json',
                          'reports/day03/validation_predictions.csv',
                          'reports/day03/observations.md'],
 'Day04 CV/selection': ['reports/day04/cv_folds.csv',
                        'reports/day04/cv_fold_summary.csv',
                        'reports/day04/cv_results.csv',
                        'reports/day04/cv_summary.csv',
                        'reports/day04/effective_config.json',
                        'reports/day04/selection.json',
                        'reports/day04/validation_predictions.csv',
                        'reports/day04/metrics.json',
                        'reports/day04/provenance.json',
                        'reports/day04/observations.md'],
 'Day05 final/error analysis': ['docs/day05_failure_cases.csv',
                                'reports/day05/validation_analysis.csv',
                                'reports/day05/validation_errors.csv',
                                'reports/day05/validation_examples.csv',
                                'reports/day05/validation_confusion_counts.csv',
                                'reports/day05/validation_confusion_counts.png',
                                'reports/day05/validation_confusion_normalized.csv',
                                'reports/day05/validation_confusion_normalized.png',
                                'reports/day05/confusion_pairs.csv',
                                'reports/day05/frozen_protocol.json',
                                'reports/day05/test_predictions.csv',
                                'reports/day05/test_metrics.json',
                                'reports/day05/test_confusion_counts.csv',
                                'reports/day05/test_confusion_counts.png',
                                'reports/day05/test_confusion_normalized.csv',
                                'reports/day05/test_confusion_normalized.png',
                                'reports/day05/final_evaluation.json',
                                'reports/day05/analysis_summary.json',
                                'reports/day05/provenance.json',
                                'reports/day05/observations.md'],
 'Day06 bundle/consistency/demo': ['models/day06/sentiment_bundle.joblib',
                                   'models/day06/manifest.json',
                                   'reports/day06/consistency.json',
                                   'docs/day06_examples.csv',
                                   'reports/day06/demo_predictions.csv']}
OPTIONAL_PATHS: list[str] = ['data/raw/FinancialPhraseBank-v1.0.zip',
 'data/raw/source.json',
 'data/interim/financial_phrasebank_75agree.csv',
 'reports/day01/quality.json',
 'reports/day01/class_distribution.csv',
 'reports/day01/text_lengths.csv',
 'reports/day01/sentiment_distribution.png',
 'reports/day01/text_length_distribution.png',
 'reports/day01/observations.md',
 'reports/day01/environment.json']
TRACKED_INPUTS: set[str] = set(['configs/dataset.json',
 'configs/experiments_day04.json',
 'configs/train_baseline.json',
 'data/raw/FinancialPhraseBank-v1.0.zip',
 'data/raw/source.json',
 'docs/day05_failure_cases.csv',
 'docs/day06_examples.csv',
 'reports/day03/observations.md',
 'reports/day04/observations.md',
 'requirements-day01.txt',
 'requirements-day03.txt',
 'requirements-dev.txt'])

required_names = {name for names in REQUIRED_GROUPS.values() for name in names}
input_names = sorted(required_names | set(OPTIONAL_PATHS))
missing_required = [name for name in sorted(required_names) if not (ROOT / name).is_file()]
missing_optional = [name for name in OPTIONAL_PATHS if not (ROOT / name).is_file()]
print("Missing required:", missing_required)
print("Missing optional:", missing_optional)
if missing_required:
    raise FileNotFoundError("Missing Day07 inputs: " + ", ".join(missing_required))
snapshot_before: dict[str, str] = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    for name in input_names if (ROOT / name).is_file()
}
inventory_rows: list[dict[str, str | bool]] = []
for name in input_names:
    inventory_rows.append({
        "path": name,
        "required": name in required_names,
        "storage": "tracked" if name in TRACKED_INPUTS else "local/ignored",
        "present": (ROOT / name).is_file(),
        "sha256": snapshot_before.get(name, "missing"),
    })
inventory = pd.DataFrame(inventory_rows)
print("Inventory:", inventory.shape, "required:", len(required_names))
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(inventory)

Missing required: []
Missing optional: []
Inventory: (63, 5) required: 53


,path,required,storage,present,sha256
0,configs/dataset.json,True,tracked,True,fc43038477b9beeb491c505244b7b0a403b8da46f062f61cdc8358126382047d
1,configs/experiments_day04.json,True,tracked,True,d9830dd3805252cc523a2fcb6fc22a28638e2c59d31feec11233cb78d908d18a
2,configs/train_baseline.json,True,tracked,True,8ffeb9230ae0cbffd6c00939694d028370ea5e825eaabf1cbb374cc025de7315
3,data/interim/financial_phrasebank_75agree.csv,False,local/ignored,True,791b78a0be391241bd6ea600fdd0128caee7c0dbe89eed10dfd00de505fb3c2e
4,data/processed/day03/splits.csv,True,local/ignored,True,b51d363cac4c00f18d760aa72d3ca6f5272becced6be45d18e763b4b9e557aaf
5,data/processed/financial_phrasebank_75agree.csv,True,local/ignored,True,39dda898a281dc3294be081917b0681524e73edb60f00913183c67714da0c448
6,data/raw/FinancialPhraseBank-v1.0.zip,False,tracked,True,0e1a06c4900fdae46091d031068601e3773ba067c7cecb5b0da1dcba5ce989a6
7,data/raw/source.json,False,tracked,True,f656fab703cbbf88174a45b0bf11b8bf0e716f32c443c1aa385e850c48d12913
8,docs/day05_failure_cases.csv,True,tracked,True,7c4c6381ea288b34c2ca90764d3a2e970550d34345d7d5712ce5b4e832f0823f
9,docs/day06_examples.csv,True,tracked,True,d99af46c74f82bf0fd999a9ea84e4de66e3ed4b7ddb9dbf93f3f6a7a8a9c363c


In [3]:
reference: InferenceReference = load_inference_reference(ROOT)
bundle: InferenceBundle = load_bundle(ROOT / "models/day06/sentiment_bundle.joblib")
assert bundle.metadata.contract == reference.contract, "Bundle/frozen reference mismatch"
assert bundle.metadata.n_train == len(reference.train)
assert bundle.metadata.n_validation == len(reference.validation)
assert bundle.metadata.n_features == reference.day05_n_features
assert reference.validation_predictions["source_row_id"].tolist() == list(reference.contract.validation_source_row_ids)
assert reference.validation_predictions["true_label_id"].tolist() == reference.validation["label_id"].tolist()
print("Candidate:", reference.contract.candidate_id)
print("Fit policy:", reference.contract.fit_policy, "seed:", reference.contract.random_state)
print("Preprocessing:", reference.contract.preprocessing_policy)
print("Class mapping:", reference.contract.label_mapping)
print("Train / validation / saved predictions:", reference.train.shape, reference.validation.shape, reference.validation_predictions.shape)
print("Recorded fitted vocabulary size:", bundle.metadata.n_features)
print("Validated frozen input/final checksums:", len(reference.contract.input_checksums))
print("Recorded replay:", bundle.metadata.validation_replay, "(not rerun)")
print("Runtime:", reference.contract.runtime_versions)
display(reference.split_summary)

Candidate: linear_svc
Fit policy: day03_train_only seed: 42
Preprocessing: minimal-v1
Class mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Train / validation / saved predictions: (2210, 5) (553, 5) (553, 3)
Recorded fitted vocabulary size: 5000
Validated frozen input/final checksums: 28
Recorded replay: exact_match (not rerun)
Runtime: {'python': '3.14.6', 'pandas': '3.0.6', 'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2', 'joblib': '1.6.0', 'threadpoolctl': '3.7.0'}


,rows,groups,negative,neutral,positive
split,,,,,
train,2210,2206,269,1374,567
validation,553,552,67,344,142
test,690,690,84,428,178


### Сохранённое evidence Day06

Ниже проверяются hashes inputs/source из существующего consistency report, его привязка к checked bundle и структура сохранённого demo. `check_inference_consistency` не вызывается: он выполняет predictions и при необходимости пишет report. Recorded parity/standalone остаются свидетельствами Day06, свежий validation replay здесь не проводится. Аналогично сравнение demo CSV не измеряет качество на новых текстах.


In [4]:
consistency = json.loads((ROOT / "reports/day06/consistency.json").read_text(encoding="utf-8"))
manifest = json.loads((ROOT / "models/day06/manifest.json").read_text(encoding="utf-8"))
if not isinstance(consistency, dict) or not isinstance(manifest, dict):
    raise ValueError("Expected JSON objects for consistency and manifest")
if consistency.get("policy") != "day06-consistency-v1":
    raise ValueError("Unexpected consistency policy")
checked_report_hashes = 0
for field in ("input_checksums", "source_checksums"):
    checksums = consistency.get(field)
    if not isinstance(checksums, dict) or not checksums:
        raise ValueError("Missing consistency checksum map: " + field)
    for name, expected in checksums.items():
        if not isinstance(name, str) or not isinstance(expected, str):
            raise ValueError("Invalid consistency checksum entry")
        file_path = (ROOT / name).resolve()
        if not file_path.is_relative_to(ROOT):
            raise ValueError("Consistency path outside project")
        assert hashlib.sha256(file_path.read_bytes()).hexdigest() == expected, name
        checked_report_hashes += 1
assert consistency.get("candidate_id") == reference.contract.candidate_id
assert consistency.get("runtime_versions") == reference.contract.runtime_versions
assert consistency.get("state_before_sha256") == consistency.get("state_after_sha256") == manifest.get("state_sha256")
print("Consistency input/source checksum entries checked:", checked_report_hashes)
print("Validated bundle state SHA-256:", manifest["state_sha256"])
print("Recorded parity:", consistency.get("parity"))
print("Recorded standalone status:", consistency.get("standalone", {}).get("status"))

demo_inputs = pd.read_csv(ROOT / "docs/day06_examples.csv", dtype=str, keep_default_na=False)
demo_outputs = pd.read_csv(ROOT / "reports/day06/demo_predictions.csv", dtype=str, keep_default_na=False)
assert demo_inputs.columns.tolist() == ["example_id", "text"]
assert demo_outputs.columns.tolist() == ["example_id", "text", "text_clean", "pred_label_id", "pred_label"]
pd.testing.assert_frame_equal(demo_inputs, demo_outputs[["example_id", "text"]])
assert len(demo_inputs) == 8
print("Saved demo inputs/outputs aligned:", demo_inputs.shape, demo_outputs.shape, "(no new predictions)")

Consistency input/source checksum entries checked: 56
Validated bundle state SHA-256: 7623ab0d4d555aed3c73944f59f85acb23e141ada2794fa826c9d7231383b7f7
Recorded parity: {'n_train': 2210, 'n_validation': 553, 'pipeline_sha256': '228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8', 'public_api_sha256': '228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8', 'reference_sha256': '228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8', 'status': 'exact_match'}
Recorded standalone status: passed
Saved demo inputs/outputs aligned: (8, 2) (8, 5) (no new predictions)


In [5]:
snapshot_after = {
    name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    for name in input_names if (ROOT / name).is_file()
}
assert snapshot_after == snapshot_before, "Day07 inputs changed during read-only audit"
print("Read-only inventory hashes unchanged:", len(snapshot_after))
print("D7-01 complete: checked reference + bundle; D7-02 and later items remain pending.")

Read-only inventory hashes unchanged: 63
D7-01 complete: checked reference + bundle; D7-02 and later items remain pending.


### Интерпретация

Checked reference содержит train `(2210, 5)`, validation `(553, 5)` и сохранённые validation predictions `(553, 3)`; TF-IDF metadata содержит `V=5000`. Это размеры исходных таблиц и fitted state, а не свежие TF-IDF matrices/predictions. Group counts берутся из проверенного split summary. Train-only policy, порядок source IDs, labels, runtime и contracts согласованы; входные hashes до/после равны.

Недостающие обязательные и опциональные inputs перечислены выше. Проверенные loaders не доказывают качество на live news, тождество утраченных весов Day05 или новый validation replay. Validation/test метрики здесь не сравниваются. `git ls-files` также выявил raw ZIP и `data/raw/source.json` как tracked вопреки описанию SPEC §3; `/data/` в `.gitignore` не исключает ранее tracked files. Это зафиксировано в docs/day07.md, index/данные не меняются. README ещё описывает Day01; SPEC §5 содержит исторические placeholders — это входы D7-02/D7-05, в D7-01 они не исправляются. Summary/ограничения и финальное demo будут отдельными пунктами.

### Самопроверка

1. Почему clone с tracked raw archive ещё не содержит готового processed dataset или model bundle? Что нужно ordinary inference?
2. Чем проверка `exists()`, SHA-256 и semantic validation отличаются друг от друга?
3. Что хранит fitted Pipeline, чего нет в saved predictions?
4. Почему чтение final cache не является новой held-out evaluation?
5. Почему `exact_match` metadata не означает, что replay выполнен в этой ячейке?
6. Что нужно отдельно проверить в D7-06, чтобы заявить запуск по README?
